# GeoForest — RF-DETR Data Pipeline

**Notebook stages:**
1. Install dependencies
2. Explore the GeoTIFF dataset
3. Generate NDVI pseudo-labels → COCO-format `annotations.json`
4. `GhanaForestDataset` — PyTorch Dataset wired to COCO annotations
5. Sanity-check: visualise boxes on sample tiles
6. DataLoader smoke-test

> **Label strategy:** We use NDVI thresholding to auto-generate bounding boxes as a
> bootstrap annotation pass. You can then manually correct in CVAT / Label Studio
> before fine-tuning, or use these pseudo-labels directly for a first training run.

## 0 — Install dependencies

In [ ]:
%%capture
!pip install rasterio supervision rfdetr scikit-image tqdm -q

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage import measure, morphology
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

warnings.filterwarnings('ignore')
print('✓ imports ok  |  torch', torch.__version__, ' | cuda:', torch.cuda.is_available())

## 1 — Paths

In [ ]:
# ── Adjust these if your dataset is mounted elsewhere ─────────────────────────
DATASET_ROOT = Path('/kaggle/input/ghana-sentinel2-forest')
GEOTIFF_DIR  = DATASET_ROOT / 'geotiffs'        # where the .tif files live
OUTPUT_DIR   = Path('/kaggle/working/geoforest')
COCO_PATH    = OUTPUT_DIR / 'annotations.json'
TILE_DIR     = OUTPUT_DIR / 'tiles'             # 512×512 PNG chips (optional)

for d in [OUTPUT_DIR, TILE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

tif_files = sorted(GEOTIFF_DIR.glob('*.tif'))
print(f'Found {len(tif_files)} GeoTIFF files')
for f in tif_files[:5]:
    print(' ', f.name)

## 2 — Explore a tile

In [ ]:
def read_tif(path):
    """
    Returns:
        bands : np.ndarray  shape (4, H, W)  float32  [B, G, R, NIR]
        meta  : dict        rasterio metadata
    """
    with rasterio.open(path) as src:
        # Read all 4 bands; shape → (bands, H, W)
        bands = src.read(out_dtype='float32')   # raw DN or surface reflectance
        meta  = src.meta.copy()
        meta['crs']       = str(src.crs)
        meta['transform'] = list(src.transform)
    return bands, meta


def compute_ndvi(bands):
    """
    bands : (4, H, W)  —  band order: Blue=0, Green=1, Red=2, NIR=3
    Returns NDVI (H, W) in [-1, 1]
    """
    red = bands[2].astype('float32')
    nir = bands[3].astype('float32')
    denom = nir + red
    ndvi  = np.where(denom == 0, 0.0, (nir - red) / denom)
    return ndvi.clip(-1, 1)


def percentile_normalise(arr, lo=2, hi=98):
    """Stretch a single-band array to [0, 1] using percentile clipping."""
    p_lo, p_hi = np.percentile(arr, [lo, hi])
    arr = (arr - p_lo) / (p_hi - p_lo + 1e-8)
    return arr.clip(0, 1)


def make_rgb(bands):
    """Returns uint8 (H, W, 3) RGB for display."""
    r = percentile_normalise(bands[2])
    g = percentile_normalise(bands[1])
    b = percentile_normalise(bands[0])
    return (np.stack([r, g, b], axis=-1) * 255).astype('uint8')


# Preview the first tile
sample_path   = tif_files[0]
bands, meta   = read_tif(sample_path)
ndvi          = compute_ndvi(bands)

print(f'Tile : {sample_path.name}')
print(f'Shape: {bands.shape}  dtype: {bands.dtype}')
print(f'NDVI : min={ndvi.min():.3f}  mean={ndvi.mean():.3f}  max={ndvi.max():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(make_rgb(bands));          axes[0].set_title('RGB');           axes[0].axis('off')
im = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[1].set_title('NDVI');                axes[1].axis('off');                fig.colorbar(im, ax=axes[1])
axes[2].hist(ndvi.ravel(), bins=80, color='#2d6a4f', edgecolor='none')
axes[2].set_title('NDVI distribution');   axes[2].set_xlabel('NDVI')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_preview.png', dpi=120)
plt.show()

## 3 — NDVI pseudo-label generation

### Strategy

We define two detection classes based on NDVI thresholds:

| Class | Class ID | NDVI range | Meaning |
|-------|----------|------------|---------|
| `forest` | 0 | ≥ 0.5 | Dense healthy vegetation |
| `degraded` | 1 | < 0.3 | Cleared / mined / bare soil |

Connected regions above/below these thresholds become bounding boxes.
Small noise blobs are filtered by minimum area.

> **Why this works as a bootstrap:** Galamsey sites strip vegetation completely
> (NDVI ≈ 0–0.1). Forest canopy sits at 0.6–0.9. The gap is large enough for
> reliable pseudo-labels on first pass.

In [ ]:
# ── Thresholds — tune these after inspecting the NDVI histogram above ─────────
NDVI_FOREST_THRESH   = 0.50   # pixels above → forest
NDVI_DEGRADED_THRESH = 0.25   # pixels below → degraded / potential galamsey
MIN_BLOB_AREA_PX     = 1024   # drop blobs smaller than this (pixels²)
MAX_BOXES_PER_CLASS  = 50     # cap per tile to avoid noise explosion

CLASS_NAMES = ['forest', 'degraded']


def ndvi_to_boxes(ndvi, label_id, mask_fn, min_area=MIN_BLOB_AREA_PX, max_boxes=MAX_BOXES_PER_CLASS):
    """
    Run connected-component analysis on a binary mask derived from `mask_fn(ndvi)`.
    Returns list of dicts: {bbox: [x, y, w, h], category_id: int, area: float}
    in COCO format (x, y = top-left corner, all in pixels).
    """
    mask   = mask_fn(ndvi).astype('uint8')
    # Optional: small morphological closing to merge nearby patches
    mask   = morphology.binary_closing(mask, footprint=morphology.disk(4)).astype('uint8')
    labels = measure.label(mask, connectivity=2)
    props  = measure.regionprops(labels)
    props  = [p for p in props if p.area >= min_area]
    # Sort by area descending, take top N
    props  = sorted(props, key=lambda p: p.area, reverse=True)[:max_boxes]

    boxes = []
    for p in props:
        min_r, min_c, max_r, max_c = p.bbox   # (row_min, col_min, row_max, col_max)
        x, y  = int(min_c), int(min_r)
        w, h  = int(max_c - min_c), int(max_r - min_r)
        boxes.append({
            'bbox':        [x, y, w, h],
            'category_id': label_id,
            'area':        float(p.area),
            'iscrowd':     0,
        })
    return boxes


print('✓ box generation helpers defined')

In [ ]:
def build_coco_annotations(tif_paths, output_json):
    """
    Iterate over all GeoTIFFs, compute NDVI pseudo-labels, and write
    a COCO-format annotations.json.
    """
    coco = {
        'info': {
            'description': 'GeoForest Ghana — NDVI pseudo-labels',
            'version': '1.0',
        },
        'categories': [
            {'id': 0, 'name': 'forest',   'supercategory': 'vegetation'},
            {'id': 1, 'name': 'degraded', 'supercategory': 'vegetation'},
        ],
        'images':      [],
        'annotations': [],
    }

    ann_id = 0

    for img_id, path in enumerate(tqdm(tif_paths, desc='Labelling tiles')):
        bands, meta = read_tif(path)
        _, H, W     = bands.shape
        ndvi        = compute_ndvi(bands)

        coco['images'].append({
            'id':        img_id,
            'file_name': path.name,
            'width':     W,
            'height':    H,
            'ndvi_mean': float(ndvi.mean()),
            'ndvi_std':  float(ndvi.std()),
        })

        forest_boxes   = ndvi_to_boxes(ndvi, label_id=0,
                                       mask_fn=lambda n: n >= NDVI_FOREST_THRESH)
        degraded_boxes = ndvi_to_boxes(ndvi, label_id=1,
                                       mask_fn=lambda n: n <  NDVI_DEGRADED_THRESH)

        for box in forest_boxes + degraded_boxes:
            box['id']       = ann_id
            box['image_id'] = img_id
            coco['annotations'].append(box)
            ann_id += 1

    with open(output_json, 'w') as f:
        json.dump(coco, f, indent=2)

    total_imgs = len(coco['images'])
    total_anns = len(coco['annotations'])
    forest_cnt  = sum(1 for a in coco['annotations'] if a['category_id'] == 0)
    degraded_cnt = total_anns - forest_cnt

    print(f'\n✓ COCO annotations saved → {output_json}')
    print(f'  Images     : {total_imgs}')
    print(f'  Annotations: {total_anns}  (forest={forest_cnt}, degraded={degraded_cnt})')
    return coco


coco_data = build_coco_annotations(tif_files, COCO_PATH)

## 4 — Visualise pseudo-labels on a sample tile

In [ ]:
def visualise_annotations(coco_data, tif_dir, img_id=0, max_boxes=30):
    img_meta = coco_data['images'][img_id]
    path     = tif_dir / img_meta['file_name']
    bands, _ = read_tif(path)
    rgb      = make_rgb(bands)
    ndvi     = compute_ndvi(bands)

    anns = [a for a in coco_data['annotations'] if a['image_id'] == img_id]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    colors    = {0: '#52b788', 1: '#e63946'}   # forest=green, degraded=red
    labels    = {0: 'forest', 1: 'degraded'}

    for ax, bg, title in zip(axes, [rgb, ndvi], ['RGB + boxes', 'NDVI + boxes']):
        if bg is ndvi:
            ax.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
        else:
            ax.imshow(bg)
        ax.set_title(f'{title} — {len(anns)} boxes', fontsize=13)
        ax.axis('off')
        for ann in anns[:max_boxes]:
            x, y, w, h = ann['bbox']
            cat        = ann['category_id']
            rect       = mpatches.Rectangle(
                (x, y), w, h,
                linewidth=1.5,
                edgecolor=colors[cat],
                facecolor='none',
                alpha=0.85,
            )
            ax.add_patch(rect)

    legend_patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in [0, 1]]
    axes[0].legend(handles=legend_patches, loc='upper right', fontsize=10)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'annotations_tile_{img_id}.png', dpi=120)
    plt.show()
    print(f'NDVI stats — mean: {img_meta["ndvi_mean"]:.3f}  std: {img_meta["ndvi_std"]:.3f}')


# Visualise first tile, and one with the lowest NDVI (most degraded)
visualise_annotations(coco_data, GEOTIFF_DIR, img_id=0)

lowest_ndvi_id = min(range(len(coco_data['images'])),
                     key=lambda i: coco_data['images'][i]['ndvi_mean'])
print(f'\nTile with lowest mean NDVI: {coco_data["images"][lowest_ndvi_id]["file_name"]}')
visualise_annotations(coco_data, GEOTIFF_DIR, img_id=lowest_ndvi_id)

## 5 — `GhanaForestDataset` — PyTorch Dataset

RF-DETR expects:
- **image**: `torch.FloatTensor` of shape `(C, H, W)` normalised to `[0, 1]`
- **target**: dict with keys `boxes` (N, 4) in `[x1, y1, x2, y2]` format, `labels` (N,)

We expose a 5-channel input: **B, G, R, NIR, NDVI** — giving the model explicit
spectral and vegetation information simultaneously.

In [ ]:
class GhanaForestDataset(Dataset):
    """
    PyTorch Dataset for GeoForest.

    Reads GeoTIFF tiles, computes NDVI as a 5th channel, and returns
    (image_tensor, target_dict) pairs compatible with RF-DETR.

    Args:
        coco_path   : path to COCO annotations.json
        tif_dir     : directory containing *.tif tiles
        transforms  : optional torchvision v2 transform pipeline
        use_ndvi_ch : if True, append NDVI as channel 5 (default True)
        classes     : list of class names in category_id order
    """

    BAND_STATS = {
        # Per-channel (B, G, R, NIR, NDVI) mean and std
        # Computed from Sentinel-2 SR typical ranges — refine after dataset-wide scan
        'mean': [0.0429, 0.0774, 0.0591, 0.2820, 0.4500],
        'std':  [0.0243, 0.0288, 0.0369, 0.1007, 0.2200],
    }

    def __init__(self, coco_path, tif_dir, transforms=None,
                 use_ndvi_ch=True, classes=None):
        with open(coco_path) as f:
            self.coco = json.load(f)

        self.tif_dir     = Path(tif_dir)
        self.transforms  = transforms
        self.use_ndvi_ch = use_ndvi_ch
        self.classes     = classes or CLASS_NAMES

        # Build fast lookup: image_id → list of annotations
        self.img_to_anns = {img['id']: [] for img in self.coco['images']}
        for ann in self.coco['annotations']:
            self.img_to_anns[ann['image_id']].append(ann)

        self.images = self.coco['images']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_meta = self.images[idx]
        path     = self.tif_dir / img_meta['file_name']

        # ── Load bands ────────────────────────────────────────────────────────
        bands, _  = read_tif(path)         # (4, H, W)  float32
        # Sentinel-2 SR values are typically in [0, 10000] or [0, 1] depending
        # on the export settings.  Normalise to [0, 1].
        if bands.max() > 10:
            bands = bands / 10000.0
        bands = bands.clip(0, 1)

        if self.use_ndvi_ch:
            ndvi  = compute_ndvi(bands)    # (H, W)
            # Rescale NDVI from [-1,1] → [0,1] for consistent channel range
            ndvi  = (ndvi + 1.0) / 2.0
            image = np.concatenate([bands, ndvi[np.newaxis]], axis=0)  # (5, H, W)
        else:
            image = bands                  # (4, H, W)

        image = torch.from_numpy(image)    # (C, H, W)  float32

        # ── Build target ──────────────────────────────────────────────────────
        anns   = self.img_to_anns[img_meta['id']]
        boxes  = []
        labels = []

        for ann in anns:
            x, y, w, h = ann['bbox']
            # Convert COCO [x, y, w, h] → [x1, y1, x2, y2]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.long)
        else:
            # Empty image — no annotations
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,),   dtype=torch.long)

        target = {
            'boxes':    boxes_t,
            'labels':   labels_t,
            'image_id': torch.tensor([img_meta['id']]),
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

    def compute_band_stats(self, max_tiles=None):
        """
        Compute dataset-wide per-channel mean and std.
        Run once and hard-code the values into BAND_STATS.
        """
        n      = min(len(self), max_tiles) if max_tiles else len(self)
        sums   = None
        sq_sum = None
        pixels = 0

        for i in tqdm(range(n), desc='Computing band stats'):
            img, _ = self[i]
            C, H, W = img.shape
            if sums is None:
                sums   = torch.zeros(C)
                sq_sum = torch.zeros(C)
            flat    = img.view(C, -1)
            sums   += flat.sum(dim=1)
            sq_sum += (flat ** 2).sum(dim=1)
            pixels += H * W

        mean = sums   / pixels
        std  = (sq_sum / pixels - mean ** 2).sqrt()
        print('\nPer-channel mean:', mean.tolist())
        print('Per-channel std: ', std.tolist())
        return mean, std


print('✓ GhanaForestDataset defined')

## 6 — Transforms

In [ ]:
# ── Custom wrapper: torchvision v2 SanitizeBoundingBoxes requires a
#    'boxes' key in a specific format. We wrap into a helper. ──────────────────

class Compose:
    """Apply a list of (image, target) → (image, target) callables."""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target


class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, target):
        if torch.rand(1) < self.p:
            _, H, W   = image.shape
            image     = image.flip(-1)    # flip width dim
            boxes     = target['boxes'].clone()
            # x1, y1, x2, y2 → flip x coords
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
            target['boxes']  = boxes
        return image, target


class RandomVerticalFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, target):
        if torch.rand(1) < self.p:
            _, H, W   = image.shape
            image     = image.flip(-2)
            boxes     = target['boxes'].clone()
            boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
            target['boxes']  = boxes
        return image, target


class Normalise:
    """Per-channel normalisation using precomputed mean/std."""
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean, dtype=torch.float32).view(-1, 1, 1)
        self.std  = torch.tensor(std,  dtype=torch.float32).view(-1, 1, 1)

    def __call__(self, image, target):
        image = (image - self.mean) / self.std
        return image, target


train_transforms = Compose([
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.5),
    Normalise(
        mean=GhanaForestDataset.BAND_STATS['mean'],
        std =GhanaForestDataset.BAND_STATS['std'],
    ),
])

val_transforms = Compose([
    Normalise(
        mean=GhanaForestDataset.BAND_STATS['mean'],
        std =GhanaForestDataset.BAND_STATS['std'],
    ),
])

print('✓ transforms defined')

## 7 — Train / val split + DataLoader

In [ ]:
from torch.utils.data import random_split

BATCH_SIZE  = 4    # Kaggle T4 GPU: 4 tiles of ~380 MB each is safe
VAL_FRAC    = 0.15
NUM_WORKERS = 2

full_ds = GhanaForestDataset(
    coco_path=COCO_PATH,
    tif_dir  =GEOTIFF_DIR,
    transforms=train_transforms,
)

n_val   = max(1, int(len(full_ds) * VAL_FRAC))
n_train = len(full_ds) - n_val

train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
# Val set uses val transforms (no flips)
val_ds.dataset.transforms = val_transforms

print(f'Train tiles: {n_train}  |  Val tiles: {n_val}')


def collate_fn(batch):
    """Stack images; keep targets as a list of dicts (variable box count)."""
    images  = torch.stack([item[0] for item in batch], dim=0)
    targets = [item[1] for item in batch]
    return images, targets


train_loader = DataLoader(
    train_ds,
    batch_size =BATCH_SIZE,
    shuffle    =True,
    num_workers=NUM_WORKERS,
    collate_fn =collate_fn,
    pin_memory =True,
)

val_loader = DataLoader(
    val_ds,
    batch_size =BATCH_SIZE,
    shuffle    =False,
    num_workers=NUM_WORKERS,
    collate_fn =collate_fn,
    pin_memory =True,
)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 8 — Smoke test: one batch through the loader

In [ ]:
images, targets = next(iter(train_loader))

print('Batch images shape :', images.shape)          # (B, C, H, W)
print('Batch images dtype :', images.dtype)
print('Batch images range :', images.min().item(), '→', images.max().item())
print()
for i, t in enumerate(targets):
    print(f'  Sample {i}: boxes={t["boxes"].shape}  labels={t["labels"].tolist()[:8]}...')

## 9 — (Optional) Compute actual dataset-wide band stats

Run this once, then hard-code the output values into `BAND_STATS` above
for proper normalisation.

In [ ]:
# Uncomment to run — takes ~2–3 min on 21 large tiles
# raw_ds = GhanaForestDataset(COCO_PATH, GEOTIFF_DIR, transforms=None)
# mean, std = raw_ds.compute_band_stats()
# # → Paste the printed values into GhanaForestDataset.BAND_STATS above

## 10 — What's next: RF-DETR model setup

The pipeline is ready. Your next notebook will:

```python
from rfdetr import RFDETRBase

model = RFDETRBase(
    num_classes=2,           # forest, degraded
    # RF-DETR expects 3-channel input by default.
    # For 5-channel (B,G,R,NIR,NDVI), you'll patch the stem conv:
    # model.model.backbone.stem[0] = nn.Conv2d(5, 32, ...)
)
model.train(
    dataset_dir=str(OUTPUT_DIR),
    epochs=50,
    batch_size=4,
)
```

**Decisions to make before training:**

| Question | Recommendation |
|----------|---------------|
| 3-ch RGB vs 5-ch multispectral | Start with 3-ch (B,G,R) to use pretrained ImageNet weights; add NIR/NDVI in fine-tune pass |
| Pseudo-labels only vs manual correction | Run a first training pass on pseudo-labels, review predictions in CVAT, correct top-N worst tiles |
| Class imbalance | Most tiles will be forest-heavy; use weighted sampling or focal loss |
| Tile size | RF-DETR works best at 640px; resize from 512 or re-export at 640 |

---
**Files written:**
- `/kaggle/working/geoforest/annotations.json` — COCO-format pseudo-labels
- `/kaggle/working/geoforest/sample_preview.png` — NDVI preview
- `/kaggle/working/geoforest/annotations_tile_*.png` — labelled tile visuals